<a href="https://colab.research.google.com/github/ced-sys/SubTerra/blob/main/Sentinel_5p_OFFL_SO2_monthly_stats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install earthengine-api

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project="subterra-466121") #set project ID

In [ ]:
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import os
from pprint import pprint

In [ ]:
START_DATE='2023-01-01'
END_DATE='2024-12-31'
SO2_BAND='SO2_column_number_density'
PRODUCT='COPERNICUS/S5P/OFFL/L3_SO2'
BUFFER_KM=5
SCALE=7000

In [ ]:
sites=[
    {'name':'Olkaria', 'lon':36.2695525, 'lat':-0.8859869, 'buffer_km':5},
]

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Google Drive mounted successfully!')

In [ ]:
def make_site_fc(sites):
  feats=[]
  for s in sites:
    geom=ee.Geometry.Point([s['lon'], s['lat']]).buffer(s.get('buffer_km', BUFFER_KM)*1000)
    feats.append(ee.Feature(geom, {'site': s['name'], 'lon':s['lon'], 'lat':s['lat']}))
  return ee.FeatureCollection(feats)

In [ ]:
site_fc=make_site_fc(sites)

In [ ]:
col=ee.ImageCollection(PRODUCT).select(SO2_BAND).filterDate(START_DATE, END_DATE)

In [ ]:
def clean_img(img):
  v=img.select(SO2_BAND)
  v_clamped=v.updateMask(v.gt(-1e-6))
  return img.addBands(v_clamped.rename('so2_clean')).copyProperties(img, ['system:time_start'])



In [ ]:
col_clean=col.map(clean_img)

In [ ]:
start_dt=dt.datetime.strptime(START_DATE, '%Y-%m-%d')
end_dt=dt.datetime.strptime(END_DATE, '%Y-%m-%d')
months=[]
cur=dt.date(start_dt.year, start_dt.month, 1)
while cur <=end_dt.date():
  ms=dt.date(cur.year, cur.month, 1)

  if cur.month==12:
    me=dt.date(cur.year+1, 1, 1)
  else:
    me=dt.date(cur.year, cur.month+1, 1)
  months.append((ms.isoformat(), me.isoformat()))

  if cur.month==12:
    cur=dt.date(cur.year+1, 1, 1)
  else:
    cur=dt.date(cur.year, cur.month+1, 1)

print(f"Number of months to process: {len(months)} (2023-2024)")

In [ ]:
rows=[]
for i, (mstart, mend) in enumerate(months):
  print(f"Processing month {i+1}/{len(months)}: {mstart} to {mend}")

  coll_m=col_clean.filterDate(mstart, mend)
  mean_img=coll_m.select('so2_clean').mean()
  max_img=coll_m.select('so2_clean').max()

  mean_stats=mean_img.reduceRegions(collection=site_fc, reducer=ee.Reducer.mean(), scale=SCALE)
  max_stats=max_img.reduceRegions(collection=site_fc, reducer=ee.Reducer.max(), scale=SCALE)

  mean_list=mean_stats.getInfo()['features']
  max_list=max_stats.getInfo()['features']

  for mf in mean_list:
    props_mean=mf['properties']
    site_name=props_mean.get('site')
    mean_val=props_mean.get('mean')

    max_val=None
    for xf in max_list:
      if xf['properties'].get('site')==site_name:
        max_val=xf['properties'].get('max')
        break

    rows.append({
        'site':site_name,
        'start':mstart,
        'end':mend,
        'so2_monthly_mean_mol_m2':mean_val,
        'so2_monthly_max_mol_m2':max_val
    })

In [ ]:
df=pd.DataFrame(rows)

df['start']=pd.to_datetime(df['start'])
df=df.sort_values(['site', 'start']).reset_index(drop=True)

In [ ]:
out_fname='Olkaria_geothermal_SO2_monthly_2023_2024.csv'
drive_path='/content/drive/MyDrive/' + out_fname
df.to_csv(drive_path, index=False)
print(f"\nSaved CSV to Google Drive: {drive_path}")
print(f"Total rows: {len(df)}")

In [ ]:
df.to_csv(out_fname, index=False)
print(f"Saved local copy: {out_fname}")

In [ ]:
print("\nFIrst few rows of data:")
print(df.head(10))

In [ ]:
for site in df['site'].unique():
  d=df[df['site']==site]
  plt.figure(figsize=(12, 5))
  plt.plot(d['start'], d['so2_monthly_mean_mol_m2'], marker='o', label='monthly mean', linewidth=2)
  plt.plot(d['start'], d['so2_monthly_max_mol_m2'], marker='x', linestyle='--', label='monthly max', linewidth=2)
  plt.title(f'SO2 time series (mol/m^2)-{site} (2023-2024)', fontsize=14, fontweight='bold' )
  plt.xlabel('Date', fontsize=12)
  plt.ylabel('SO2 column (mol/m^2)', fontsize=12)
  plt.legend(fontsize=11)
  plt.grid(alpha=0.3)
  plt.tight_layout()
  plt.show()